# Run a VCell PDE simulation from solver input files
1. Copy all files from solver_input directory to a temporary directory for solving
2. prepare empty solver output directory, copying in the functions file
2. Execute the Finite Volume PDE solver

### In this example, we are using the Model instance as an example, as the get input files endpoint is not yet implemented.

In [ ]:
from pathlib import Path

import os

from pyvcell.sbml.sbml_simulation import SbmlSpatialSimulation
from pyvcell.sbml.sbml_spatial_model import SbmlSpatialModel


model_fp = Path(os.getcwd()).parent / "models" / "TinySpatialProject_Application0.sbml"

# define editable spatial model and simulation instances
model = SbmlSpatialModel(filepath=model_fp)
simulation = SbmlSpatialSimulation(sbml_model=model)
result = simulation.run()

In [ ]:
from pyvcell.sim_results.vtk_data import VtkData

vtk_data: VtkData = result.vtk_data
dir(vtk_data)

In [ ]:
from pyvcell.sim_results.plotter import Plotter

plotter: Plotter = result.plotter
dir(plotter)

In [ ]:
def get_plottable_indices() -> list[int]:
    import numpy as np

    nchannels = result.zarr_dataset.shape[1]
    valid = []
    for n, i in enumerate(list(range(nchannels))):
        x = result.zarr_dataset[0, n, :, :, :]
        any = np.any(x)
        valid.append(i) if any else None

    return list(set(valid))

In [ ]:
indices = get_plottable_indices()

indices, result.zarr_dataset.shape

In [ ]:
ids = result.get_channel_ids()
s1 = result.get_channel('s1')
time_index = 3
data = result.get_slice(s1.label, time_index)

In [ ]:
data.ndim

In [ ]:
channels = result.zarr_dataset.attrs.asdict()["metadata"].get("channels")

In [ ]:
result.zarr_dataset.shape

In [ ]:
x = [1, 4, 5, 3, 6, 35]
y = list(filter(
    lambda v: v % 2 == 0,
    x
))

In [ ]:


from pyvcell.sim_results.result import Result
from pyvcell.sim_results.zarr_types import ChannelMetadata


def get_channel(result: Result, label: str) -> ChannelMetadata:
    getter = filter(lambda c: c.label == label, result.channel_data)
    c = next(getter, None)

    if c is None:
        raise ValueError(f"No channel found with label '{label}'")

    if next(getter, None) is not None:
        raise ValueError(f"More than one '{label}' channel found")

    return c


chan = get_channel(result, 's0')
chan

In [ ]:
import numpy as np
channel_id = chan.label
np.array([result.get_slice(channel_id, int(t)) for t in result.time_points])

In [ ]:
next(iter([1, 2, 3]))

In [ ]:
[(channel.label, channel.min_values) for channel in result.metadata.channels]

In [ ]:

time_index = 0
channel_id = "s0"
z_index = 0

In [ ]:
plotter.plot_slice_2d(time_index, channel_id, z_index)

In [ ]:
plotter.plot_slice_3d(time_index, channel_id)
# result.plot_slice_3d(time_index, channel_index)

In [ ]:
plotter.plot_concentrations()
# result.plot_concentrations()

|## extract the vcell simulation dataset from the tarball (compressed to save space)

## read vcell simulation results metadata
* `PdeDataSet` contains the metadata for the tabular simulation results (e.g. state variables, shape, time points)
* `DataFunctions` contains the function definitions (name, expression, type, domain)
![vcell simulation results](./example_vcell_ui.png)

## write the vcell simulation dataset to zarr including:
* metadata
* numerical datasets from stored data and evaluated functions
* ... masks for domains coming soon (e.g. cell, extracellular, etc.)

## Open and display slices from the zarr dataset as an image
* no masking for domains
* different colormap and scaling

## Open and display slices from the post processing dataset as an image

In [ ]:
len(result.channel_data)

In [ ]:


result.plotter.animate_channel_3d(channel_id)

In [ ]:
result.channel_data

In [ ]:
nchannels = result.zarr_dataset.shape[1]
len(result.channel_data)

In [ ]:
result.metadata.mesh

@

In [ ]:
# # animate fluor dataset image over time
# fluor_index = 4
# result.plotter.animate_image(fluor_index)

## Open and display Variable Statistics from the post processing dataset

In [ ]:
post_processing = result.post_processing
post_processing.variables[0].stat_var_unit

In [ ]:
plotter.plot_averages()

In [ ]:
x